# 01 · Data cleaning and preprocessing

**Project:** Sales & Business Performance Analytics (Superstore)
**Input:** `data/raw/sample_-_superstore.xls` (never modified)
**Outputs:**
- `data/interim/orders_clean.csv`: every row after cleaning, with flag columns
- `data/processed/orders_analysis.csv`: the rows used in the EDA and KPI notebooks (United States only)
- `reports/data_quality_checks.csv`, `reports/cleaning_log.csv` and `reports/data_dictionary.csv`

All cleaning rules live in `src/cleaning.py`, so the steps can be repeated outside this notebook. The last code cell recomputes every number quoted in the summary at the end.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the project root (the folder holding src/ and data/) and make src importable
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir() and (p / "data").is_dir())
sys.path.insert(0, str(ROOT))
from src import cleaning as cl
from src.checks import Claims

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

RAW_PATH = ROOT / "data" / "raw" / cl.RAW_FILE_NAME
print("Project root:", ROOT.name)
print("Raw file    :", RAW_PATH.relative_to(ROOT))

Project root: superstore-sales-analytics
Raw file    : data/raw/sample_-_superstore.xls


## 1. Load the raw workbook

The workbook has three sheets: **Orders** (the transactions), **Returns** (returned Order IDs) and **People** (regional managers, not used). Reading a legacy `.xls` needs the `xlrd` package.

In [2]:
raw_orders, returns = cl.load_raw(RAW_PATH)
print("Orders sheet :", raw_orders.shape)
print("Returns sheet:", returns.shape)
raw_orders.head()

Orders sheet : (10194, 21)
Returns sheet: (296, 2)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,State/Province,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,US-2023-103800,2023-01-03,2023-01-07,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Forms/Pg., 200 Dupl. Sets/Book",16.448,2,0.2,5.5512
1,2,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,3.540,2,0.8,-5.4870
2,3,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,11.784,3,0.2,4.2717
3,4,US-2023-112326,2023-01-04,2023-01-08,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,272.736,3,0.2,-64.7748
4,5,US-2023-141817,2023-01-05,2023-01-12,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,19143,East,OFF-AR-10003478,Office Supplies,Art,"Avery Hi-Liter EverBold Pen Style Fluorescent Highlighters, 4/Pack",19.536,3,0.2,4.8840


## 2. Structure, types and coverage

In [3]:
raw_orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Row ID          10194 non-null  int64         
 1   Order ID        10194 non-null  str           
 2   Order Date      10194 non-null  datetime64[us]
 3   Ship Date       10194 non-null  datetime64[us]
 4   Ship Mode       10194 non-null  str           
 5   Customer ID     10194 non-null  str           
 6   Customer Name   10194 non-null  str           
 7   Segment         10194 non-null  str           
 8   Country/Region  10194 non-null  str           
 9   City            10194 non-null  str           
 10  State/Province  10194 non-null  str           
 11  Postal Code     10194 non-null  object        
 12  Region          10194 non-null  str           
 13  Product ID      10194 non-null  str           
 14  Category        10194 non-null  str           
 15  Sub-Category 

In [4]:
print("Missing values       :", int(raw_orders.isna().sum().sum()))
print("Row ID unique        :", raw_orders["Row ID"].is_unique)
print("Rows                 :", f"{len(raw_orders):,}")
print("Distinct Order IDs   :", f"{raw_orders['Order ID'].nunique():,}  -> each row is an order LINE, not an order")
lines_per_order = raw_orders.groupby("Order ID").size()
print("Lines per order      : min", lines_per_order.min(), "| median", int(lines_per_order.median()), "| max", lines_per_order.max())
print("Postal Code as read  :", raw_orders["Postal Code"].map(lambda v: type(v).__name__).value_counts().to_dict(),
      "-> numbers and text mixed")

display(pd.DataFrame({"Type": raw_orders.dtypes.astype(str), "Distinct values": raw_orders.nunique()}))
display(raw_orders[["Sales", "Quantity", "Discount", "Profit"]].describe().round(3))

Missing values       : 0
Row ID unique        : True
Rows                 : 10,194
Distinct Order IDs   : 5,111  -> each row is an order LINE, not an order
Lines per order      : min 1 | median 1 | max 14
Postal Code as read  : {'int': 9983, 'str': 211} -> numbers and text mixed


,Type,Distinct values
Row ID,int64,10194
Order ID,str,5111
Order Date,datetime64[us],1242
Ship Date,datetime64[us],1338
Ship Mode,str,4
Customer ID,str,804
Customer Name,str,800
Segment,str,3
Country/Region,str,2
City,str,542


,Sales,Quantity,Discount,Profit
count,10194.000,10194.000,10194.000,10194.000
mean,228.226,3.792,0.155,28.673
std,619.907,2.228,0.206,232.465
min,0.444,1.000,0.000,-6599.978
25%,17.220,2.000,0.000,1.761
50%,53.910,3.000,0.200,8.690
75%,209.500,5.000,0.200,29.298
max,22638.480,14.000,0.800,8399.976


In [5]:
dated = cl.parse_dates(raw_orders)
months = dated["Order Date"].dt.to_period("M")
print("Order dates :", dated["Order Date"].min().date(), "to", dated["Order Date"].max().date())
print("Months with orders:", months.nunique(), "of", len(pd.period_range(months.min(), months.max(), freq="M")))
future = dated["Order Date"] > cl.REFERENCE_DATE
print(f"Lines dated after the analysis date ({cl.REFERENCE_DATE.date()}): {future.sum():,}")

coverage = (dated.groupby(dated["Order Date"].dt.year)
                 .agg(Lines=("Sales", "size"), Orders=("Order ID", "nunique"), Sales=("Sales", "sum"),
                      Future_dated_lines=("Order Date", lambda d: int((d > cl.REFERENCE_DATE).sum()))))
coverage.index.name = "Order Year"
display(coverage.round(0))
print("Rows by country:", dated["Country/Region"].value_counts().to_dict())

Order dates : 2023-01-03 to 2026-12-30
Months with orders: 48 of 48
Lines dated after the analysis date (2026-10-15): 1,093


,Lines,Orders,Sales,Future_dated_lines
Order Year,,,,
2023,2051,995,494040.0,0
2024,2130,1053,472993.0,0
2025,2634,1340,613934.0,0
2026,3379,1723,745568.0,1093


Rows by country: {'United States': 9994, 'Canada': 200}


## 3. Data-quality checks

Each check runs on **all rows** and on **United States rows only**. Consistency checks test that every line of an order has the same date, ship mode, customer and location; that IDs follow their patterns; and that each product's price is consistent. A price is consistent when the implied list price (Sales ÷ Quantity ÷ (1 − Discount)) is within 5% of the same product's price on US lines.

In [6]:
qc = cl.quality_checks(dated, returns)
display(qc)

must_be_zero = qc["Group"].isin(["Completeness", "Consistency", "Value range"])
print("Consistency, completeness and range checks failing on ALL rows:", int((qc.loc[must_be_zero, "All rows"] > 0).sum()))
print("Consistency, completeness and range checks failing on US rows :", int((qc.loc[must_be_zero, "US rows"] > 0).sum()))

,Group,Check,Unit,All rows,US rows,Detail
0,Completeness,Missing values,cells,0,0,
1,Duplicates,Repeated Row IDs,rows,0,0,
2,Duplicates,Lines repeating another line (all columns except Row ID),lines,2,1,the second copy of each pair
3,Consistency,Orders with more than one Order Date,orders,0,0,
4,Consistency,Orders with more than one Ship Date,orders,0,0,
5,Consistency,Orders with more than one Ship Mode,orders,0,0,
6,Consistency,Orders with more than one Customer ID,orders,2,0,
7,Consistency,Orders with more than one Segment,orders,0,0,
8,Consistency,Orders with more than one Country/Region,orders,0,0,
9,Consistency,Orders with more than one City,orders,2,0,


Consistency, completeness and range checks failing on ALL rows: 6
Consistency, completeness and range checks failing on US rows : 0


## 4. Why the Canadian rows are excluded

Every consistency failure above comes from the 200 Canadian rows. US rows pass all of them. The cells below show the evidence.

In [7]:
is_ca = dated["Country/Region"].eq("Canada")
ca = dated[is_ca]
print(f"Canadian rows: {len(ca)} lines, {ca['Order ID'].nunique()} orders, "
      f"{ca['Sales'].sum() / dated['Sales'].sum():.2%} of all sales")

# 4a. Prices that belong to a different product
lp = cl.list_price(dated)
key = ["Product ID", "Product Name"]
us_price = lp[~is_ca].groupby([dated.loc[~is_ca, k] for k in key]).median().rename("US list price")
mismatch = cl.price_mismatch(dated)
examples = (dated[mismatch].assign(**{"Line list price": lp[mismatch]})
            .join(us_price, on=key)
            .assign(Ratio=lambda t: t["Line list price"] / t["US list price"]))
print(f"\nCanadian lines priced >5% away from the product's US price: {int(mismatch[is_ca].sum())} of {len(ca)} "
      f"({mismatch[is_ca].mean():.0%}); US lines: {int(mismatch[~is_ca].sum())}")
display(examples.reindex(examples["Ratio"].sub(1).abs().sort_values(ascending=False).index)
        [["Row ID", "Order ID", "Product Name", "Sales", "Quantity", "Discount", "Line list price", "US list price"]]
        .head(8).round(2))

Canadian rows: 200 lines, 102 orders, 1.26% of all sales

Canadian lines priced >5% away from the product's US price: 74 of 200 (37%); US lines: 0


,Row ID,Order ID,Product Name,Sales,Quantity,Discount,Line list price,US list price
9010,9011,CA-2026-131807,Staples,5199.96,4,0.0,1299.99,2.98
936,937,CA-2023-115777,Xerox 200,387.99,1,0.0,387.99,6.48
8385,8386,CA-2026-115777,Xerox 200,387.99,1,0.0,387.99,6.48
3901,3902,CA-2024-119508,"Eldon Image Series Desk Accessories, Burgundy",361.76,2,0.1,200.98,4.18
5180,5181,CA-2025-112109,Wilson Jones Active Use Binders,863.64,9,0.2,119.95,7.28
5012,5013,CA-2025-121465,GBC ProClick Spines for 32-Hole Punch,310.12,2,0.0,155.06,12.53
7194,7195,CA-2026-130494,GBC ProClick Spines for 32-Hole Punch,310.12,2,0.0,155.06,12.53
6585,6586,CA-2025-141082,Avery 517,79.97,3,0.4,44.43,3.69


In [8]:
# 4b. Canadian lines that copy a US line (same customer, date, products and values; different city)
copies = cl.copies_of_us_lines(dated)
print(f"Canadian lines identical to a US line: {len(copies)} lines in {copies['Order ID'].nunique()} orders")
display(copies.groupby(["Order ID", "Order ID (US)", "City"]).agg(Lines=("Sales", "size"), Sales=("Sales", "sum")).round(2))

# 4c. Order numbers reused in another year (the same order appearing twice)
parts = dated.drop_duplicates("Order ID")["Order ID"].str.extract(r"^(?P<prefix>[A-Z]{2})-(?P<year>\d{4})-(?P<num>\d+)$")
reuse = parts.groupby(["prefix", "num"])["year"].nunique().gt(1).groupby(level="prefix").agg(["sum", "size"])
reuse.columns = ["Numbers used in more than one year", "Distinct order numbers"]
display(reuse)
print(dated.loc[dated["Order ID"].str.endswith("-131807"), ["Order ID", "Order Date", "City", "Product ID", "Sales"]])

# 4d. One customer with five IDs, the IDs changing line by line inside one order
print(dated.loc[dated["Customer Name"].eq("Harry Olson"), ["Order ID", "Customer ID", "Customer Name", "Product ID", "Sales"]])

Canadian lines identical to a US line: 14 lines in 6 orders


,,,Lines,Sales
Order ID,Order ID (US),City,,
CA-2023-152443,US-2023-152443,Montreal,3,711.68
CA-2023-166086,US-2023-166086,Toronto,4,1091.35
CA-2024-139675,US-2024-139675,Calgary,2,1242.90
CA-2024-152681,US-2024-152681,Montreal,1,6.68
CA-2024-158792,US-2024-158792,Montreal,1,22.20
CA-2026-166695,US-2026-166695,Vancouver,3,868.45


,Numbers used in more than one year,Distinct order numbers
prefix,,
CA,29,63
US,0,5009


            Order ID Order Date      City       Product ID     Sales
1293  CA-2023-131807 2023-10-05   Calgary  FUR-CH-10001190   435.170
1294  CA-2023-131807 2023-10-05  Edmonton  FUR-FU-10004666    72.900
1295  CA-2023-131807 2023-10-05  Edmonton  FUR-TA-10004086   206.350
1296  CA-2023-131807 2023-10-05  Edmonton  OFF-AP-10004052    15.800
1297  CA-2023-131807 2023-10-05   Calgary  OFF-FA-10002780    14.900
1298  CA-2023-131807 2023-10-05  Edmonton  TEC-PH-10000702     7.990
8991  US-2026-131807 2026-10-05     Chico  FUR-CH-10001190   435.168
8993  US-2026-131807 2026-10-05     Chico  FUR-FU-10004666    72.900
8994  US-2026-131807 2026-10-05     Chico  FUR-TA-10004086   206.352
8995  US-2026-131807 2026-10-05     Chico  OFF-AP-10004052    15.800
8999  US-2026-131807 2026-10-05     Chico  OFF-FA-10002780    14.900
9003  US-2026-131807 2026-10-05     Chico  TEC-PH-10000702     7.992
9006  CA-2026-131807 2026-10-05   Calgary  FUR-CH-10001190   221.382
9007  CA-2026-131807 2026-10-05  E

In [9]:
# 4e. Effect on the analysis: the Canadian rows are 1.3% of sales but distort product results
def subcat(d):
    t = d.groupby("Sub-Category")["Sales"].sum()
    return pd.DataFrame({"Sales": t, "Rank": t.rank(ascending=False).astype(int)})

effect = subcat(dated[~is_ca]).join(subcat(dated), lsuffix=" (US only)", rsuffix=" (with Canada)")
effect["Change %"] = (effect["Sales (with Canada)"] / effect["Sales (US only)"] - 1) * 100
display(effect.sort_values("Change %", ascending=False).head(5).round(1))
print("Largest Canadian line:", dated.loc[ca["Sales"].idxmax(), ["Order ID", "Product Name", "Quantity", "Sales"]].to_dict())

,Sales (US only),Rank (US only),Sales (with Canada),Rank (with Canada),Change %
Sub-Category,,,,,
Fasteners,3024.3,17,8532.2,17,182.1
Furnishings,91705.2,11,95598.1,11,4.2
Chairs,328449.1,2,335768.2,1,2.2
Art,27118.8,14,27659.0,14,2.0
Binders,203412.7,5,207354.9,5,1.9


Largest Canadian line: {'Order ID': 'CA-2026-131807', 'Product Name': 'Staples', 'Quantity': 4, 'Sales': 5199.96}


**Decision:** all analysis uses the **United States rows only**. The Canadian rows stay in `orders_clean.csv` with `in_scope = False`, so the decision can be reviewed or reversed, and the KPI notebook shows how the totals change if they are included.

## 5. Apply the cleaning rules

| Rule | Why |
|---|---|
| Trim text and collapse repeated spaces | 7 product names contain double spaces |
| Upper-case Customer IDs | 3 IDs contain lowercase letters (e.g. `Dl-13600`); checked that this merges no two customers |
| Postal Code as text; US ZIP codes back to 5 digits | Excel stored ZIP codes as numbers, so 438 US codes lost their leading zero |
| Remove exact repeat lines | 2 lines repeat another line in every column except Row ID |
| Add Order Year, Order Quarter, Order Month, Shipping Days | Needed for time analysis and shipping times |
| Flag `is_returned`, `is_future_dated`, `in_scope` | Keep returns, future-dated lines and Canadian rows visible instead of deleting them |

Nothing else is changed. No missing values exist, so none are filled, and no value is estimated.

In [10]:
clean_df, cleaning_log = cl.clean(raw_orders, returns)
analysis = cl.analysis_set(clean_df)
display(cleaning_log)
print("Cleaned rows  :", clean_df.shape)
print("Analysis rows :", analysis.shape, f"-> {analysis['Order ID'].nunique():,} orders, {analysis['Customer ID'].nunique():,} customers")

,Step,Rows before,Rows after,Values changed,Note
0,Trim text and collapse repeated spaces,10194,10194,32,cosmetic; Product Name: 32
1,Customer ID to upper case,10194,10194,43,"Co-12640, Dl-13600, Dp-13240"
2,Postal Code to text; leading zero restored on US ZIP codes,10194,10194,438,"states: Connecticut, Maine, Massachusetts, New Hampshire, New Jersey, Rhode Island"
3,Remove exact repeat lines,10194,10192,2,"removed Row IDs 392 (US-2023-150119), 1700 (CA-2023-153623)"
4,"Add Order Year, Order Quarter, Order Month, Shipping Days",10192,10192,0,"Shipping Days = Ship Date - Order Date (order to shipment, not delivery)"
5,Flag lines of returned orders (is_returned),10192,10192,800,296 orders in the Returns sheet
6,Flag lines dated after the analysis date (is_future_dated),10192,10192,1093,Order Date after 2026-10-15
7,Flag analysis scope (in_scope = United States),10192,10192,199,Canadian rows set to out of scope


Cleaned rows  : (10192, 28)
Analysis rows : (9993, 28) -> 5,009 orders, 793 customers


In [11]:
# Before / after examples
zip_fixed = raw_orders["Postal Code"].astype(str).str.fullmatch(r"\d{1,4}") & raw_orders["Country/Region"].eq("United States")
ex = raw_orders.loc[zip_fixed, ["Row ID", "City", "State/Province", "Postal Code"]].head(3)
display(ex.merge(clean_df[["Row ID", "Postal Code"]], on="Row ID", suffixes=(" before", " after")))

ids = raw_orders.loc[raw_orders["Customer ID"].str.contains("[a-z]", regex=True), ["Customer ID", "Customer Name"]].drop_duplicates()
display(ids.assign(**{"Customer ID after": ids["Customer ID"].str.upper()}))

names = raw_orders.loc[raw_orders["Product Name"].str.contains("  ", regex=False), "Product Name"].drop_duplicates().head(3)
display(pd.DataFrame({"Before": names.map(repr), "After": names.str.replace(r"\s{2,}", " ", regex=True).map(repr)}))

,Row ID,City,State/Province,Postal Code before,Postal Code after
0,99,Linden,New Jersey,7036,07036
1,291,Revere,Massachusetts,2151,02151
2,293,Revere,Massachusetts,2151,02151


,Customer ID,Customer Name,Customer ID after
276,Dl-13600,Dorris liebe,DL-13600
534,Dp-13240,Dean percer,DP-13240
632,Co-12640,Corey-Lock,CO-12640


,Before,After
41,"'Tyvek Top-Opening Peel & Seel Envelopes, Plain White'","'Tyvek Top-Opening Peel & Seel Envelopes, Plain White'"
796,'Eldon Mobile Mega Data Cart Mega Stackable Add-On Trays','Eldon Mobile Mega Data Cart Mega Stackable Add-On Trays'
917,'Avery Heavy-Duty EZD Binder With Locking Rings','Avery Heavy-Duty EZD Binder With Locking Rings'


## 6. Save the outputs

In [12]:
clean_path = ROOT / cl.CLEAN_FILE
analysis_path = ROOT / cl.ANALYSIS_FILE
reports = ROOT / "reports"
reports.mkdir(exist_ok=True)

cl.save_csv(clean_df, clean_path)
cl.save_csv(analysis, analysis_path)
qc.to_csv(reports / "data_quality_checks.csv", index=False)
cleaning_log.to_csv(reports / "cleaning_log.csv", index=False)
cl.DATA_DICTIONARY.to_csv(reports / "data_dictionary.csv", index=False)

# Read the files back: they must match the in-memory tables exactly
round_trip_ok = cl.load_clean(clean_path).equals(clean_df) and cl.load_clean(analysis_path).equals(analysis)
print("Saved:", clean_path.relative_to(ROOT), "|", analysis_path.relative_to(ROOT))
print("Files read back identical:", round_trip_ok)

Saved: data/interim/orders_clean.csv | data/processed/orders_analysis.csv
Files read back identical: True


## 7. Data dictionary

In [13]:
cl.DATA_DICTIONARY

,Column,Type,Description
0,Row ID,integer,Source line number; unique.
1,Order ID,text,"Order identifier, e.g. US-2023-103800 (country-year-number). One order can have severa..."
2,Order Date,date,Date the order was placed.
3,Ship Date,date,Date the order was shipped.
4,Ship Mode,text,"Same Day, First Class, Second Class or Standard Class."
5,Customer ID,text,Customer identifier; upper-cased during cleaning.
6,Customer Name,text,Customer name as provided.
7,Segment,text,"Consumer, Corporate or Home Office."
8,Country/Region,text,United States or Canada (only United States rows are analysed).
9,City,text,Shipping city.


## 8. Reconciliation and claim verification

Every raw line is accounted for: analysis rows + out-of-scope (Canadian) rows + removed repeats = the raw sheet, for lines, sales, profit and quantity.

In [14]:
removed = raw_orders[~raw_orders["Row ID"].isin(clean_df["Row ID"])]
out_of_scope = clean_df[~clean_df["in_scope"]]
recon = pd.DataFrame({
    "Raw sheet": [len(raw_orders), raw_orders["Sales"].sum(), raw_orders["Profit"].sum(), raw_orders["Quantity"].sum()],
    "Analysis set": [len(analysis), analysis["Sales"].sum(), analysis["Profit"].sum(), analysis["Quantity"].sum()],
    "Out of scope (Canada)": [len(out_of_scope), out_of_scope["Sales"].sum(), out_of_scope["Profit"].sum(), out_of_scope["Quantity"].sum()],
    "Removed repeats": [len(removed), removed["Sales"].sum(), removed["Profit"].sum(), removed["Quantity"].sum()],
}, index=["Lines", "Sales", "Profit", "Quantity"])
recon["Difference"] = recon["Raw sheet"] - recon[["Analysis set", "Out of scope (Canada)", "Removed repeats"]].sum(axis=1)
display(recon.round(2))

,Raw sheet,Analysis set,Out of scope (Canada),Removed repeats,Difference
Lines,10194.00,9993.00,199.00,2.00,0.0
Sales,2326534.35,2296919.49,29234.37,380.49,-0.0
Profit,292296.81,286409.08,5864.38,23.36,0.0
Quantity,38654.00,37871.00,773.00,10.00,0.0


In [15]:
c = Claims()
q = qc.set_index("Check")

# Raw structure
c.check("Orders sheet: 10,194 rows x 21 columns; Returns sheet: 296 rows", (raw_orders.shape, len(returns)) == ((10194, 21), 296), (raw_orders.shape, len(returns)))
c.check("No missing values; Row ID unique", raw_orders.isna().sum().sum() == 0 and raw_orders["Row ID"].is_unique, int(raw_orders.isna().sum().sum()))
c.check("5,111 distinct Order IDs, so rows are order lines", raw_orders["Order ID"].nunique() == 5111, raw_orders["Order ID"].nunique())
c.check("Order dates 3 Jan 2023 to 30 Dec 2026, all 48 months present",
        (str(dated["Order Date"].min().date()), str(dated["Order Date"].max().date()), months.nunique()) == ("2023-01-03", "2026-12-30", 48),
        (str(dated["Order Date"].min().date()), str(dated["Order Date"].max().date()), months.nunique()))
c.check("1,093 raw lines dated after 15 Oct 2026", int(future.sum()) == 1093, int(future.sum()))

# Quality checks
c.check("US rows pass every completeness, consistency and value-range check", (qc.loc[must_be_zero, "US rows"] == 0).all(),
        qc.loc[must_be_zero & (qc["US rows"] > 0), "Check"].tolist())
c.check("2 exact repeat lines (Row IDs 392 and 1700), 1 of them US",
        (q.loc["Lines repeating another line (all columns except Row ID)", "All rows"], q.loc["Lines repeating another line (all columns except Row ID)", "US rows"],
         sorted(removed["Row ID"])) == (2, 1, [392, 1700]), sorted(removed["Row ID"]))

# Canada evidence
c.check("Canada: 200 rows, 102 orders, 1.26% of sales", (len(ca), ca["Order ID"].nunique(), round(ca["Sales"].sum() / dated["Sales"].sum() * 100, 2)) == (200, 102, 1.26),
        (len(ca), ca["Order ID"].nunique(), round(ca["Sales"].sum() / dated["Sales"].sum() * 100, 3)))
c.check("74 Canadian lines (37%) priced >5% away from the US price; 0 US lines", (int(mismatch[is_ca].sum()), round(mismatch[is_ca].mean() * 100), int(mismatch[~is_ca].sum())) == (74, 37, 0),
        (int(mismatch[is_ca].sum()), round(mismatch[is_ca].mean() * 100, 1), int(mismatch[~is_ca].sum())))
c.check("14 Canadian lines in 6 orders copy US lines", (len(copies), copies["Order ID"].nunique()) == (14, 6), (len(copies), copies["Order ID"].nunique()))
c.check("29 of 63 Canadian order numbers reused in another year; 0 US", (int(reuse.loc["CA"].iloc[0]), int(reuse.loc["CA"].iloc[1]), int(reuse.loc["US"].iloc[0])) == (29, 63, 0), reuse.to_dict("index"))
c.check("2 Canadian orders with more than one Customer ID and 2 with more than one City",
        (q.loc["Orders with more than one Customer ID", "All rows"], q.loc["Orders with more than one City", "All rows"]) == (2, 2),
        (q.loc["Orders with more than one Customer ID", "All rows"], q.loc["Orders with more than one City", "All rows"]))
c.check("Harry Olson appears under 5 Customer IDs", dated.loc[dated["Customer Name"].eq("Harry Olson"), "Customer ID"].nunique() == 5,
        dated.loc[dated["Customer Name"].eq("Harry Olson"), "Customer ID"].nunique())
c.check("With Canada, Fasteners sales rise 182% and Chairs overtakes Phones as the top sub-category",
        round(effect.loc["Fasteners", "Change %"]) == 182 and effect.loc["Chairs", "Rank (with Canada)"] == 1 and effect.loc["Phones", "Rank (US only)"] == 1,
        (round(effect.loc["Fasteners", "Change %"], 1), effect["Rank (with Canada)"].idxmin(), effect["Rank (US only)"].idxmin()))
c.check("Largest Canadian line: Staples, 4 units, $5,199.96", (ca.loc[ca["Sales"].idxmax(), "Product Name"], round(ca["Sales"].max(), 2)) == ("Staples", 5199.96),
        (ca.loc[ca["Sales"].idxmax(), "Product Name"], round(ca["Sales"].max(), 2)))

# Cleaning rules
zip_states = sorted(raw_orders.loc[zip_fixed, "State/Province"].unique())
c.check("438 US ZIP codes missing a leading zero, in CT, ME, MA, NH, NJ and RI",
        int(zip_fixed.sum()) == 438 and zip_states == ["Connecticut", "Maine", "Massachusetts", "New Hampshire", "New Jersey", "Rhode Island"], (int(zip_fixed.sum()), zip_states))
c.check("Linden, New Jersey: 7036 becomes 07036",
        set(clean_df.loc[(clean_df["City"] == "Linden") & (clean_df["State/Province"] == "New Jersey"), "Postal Code"]) == {"07036"},
        set(clean_df.loc[(clean_df["City"] == "Linden") & (clean_df["State/Province"] == "New Jersey"), "Postal Code"]))
c.check("3 lowercase Customer IDs on 43 lines", (len(ids), int(raw_orders["Customer ID"].str.contains("[a-z]", regex=True).sum())) == (3, 43),
        (len(ids), int(raw_orders["Customer ID"].str.contains("[a-z]", regex=True).sum())))
c.check("7 product names with repeated spaces; collapsing them merges no names",
        (q.loc["Product names containing repeated spaces", "All rows"], clean_df["Product Name"].nunique()) == (7, raw_orders["Product Name"].nunique()),
        (q.loc["Product names containing repeated spaces", "All rows"], clean_df["Product Name"].nunique(), raw_orders["Product Name"].nunique()))
c.check("32 Product IDs cover more than one product name", q.loc["Product IDs used for more than one product name", "All rows"] == 32, q.loc["Product IDs used for more than one product name", "All rows"])
c.check("296 returned orders, all found in Orders and all US; 800 lines", (returns["Order ID"].nunique(), q.loc["Returned Order IDs not found in Orders", "All rows"],
        bool(returns["Order ID"].str.startswith("US-").all()), int(clean_df["is_returned"].sum())) == (296, 0, True, 800),
        (returns["Order ID"].nunique(), int(clean_df["is_returned"].sum())))

# Outputs
c.check("Cleaned file: 10,192 rows x 28 columns", clean_df.shape == (10192, 28), clean_df.shape)
c.check("Analysis set: 9,993 lines, 5,009 orders, 793 customers", (len(analysis), analysis["Order ID"].nunique(), analysis["Customer ID"].nunique()) == (9993, 5009, 793),
        (len(analysis), analysis["Order ID"].nunique(), analysis["Customer ID"].nunique()))
c.check("1,069 analysis lines dated after 15 Oct 2026", int(analysis["is_future_dated"].sum()) == 1069, int(analysis["is_future_dated"].sum()))
c.check("Reconciliation: no difference for lines, sales, profit or quantity", (recon["Difference"].abs() < 1e-6).all(), recon["Difference"].round(9).to_dict())
c.check("Saved CSV files read back identical", round_trip_ok, round_trip_ok)

c.report()

26 of 26 claims verified against the data.


,Claim,Passed,Computed
0,"Orders sheet: 10,194 rows x 21 columns; Returns sheet: 296 rows",True,"((10194, 21), 296)"
1,No missing values; Row ID unique,True,0
2,"5,111 distinct Order IDs, so rows are order lines",True,5111
3,"Order dates 3 Jan 2023 to 30 Dec 2026, all 48 months present",True,"('2023-01-03', '2026-12-30', 48)"
4,"1,093 raw lines dated after 15 Oct 2026",True,1093
5,"US rows pass every completeness, consistency and value-range check",True,[]
6,"2 exact repeat lines (Row IDs 392 and 1700), 1 of them US",True,"[392, 1700]"
7,"Canada: 200 rows, 102 orders, 1.26% of sales",True,"(200, 102, 1.261)"
8,74 Canadian lines (37%) priced >5% away from the US price; 0 US lines,True,"(74, 37.0, 0)"
9,14 Canadian lines in 6 orders copy US lines,True,"(14, 6)"


## Summary: data cleaning

- **Source:** the Orders sheet has 10,194 rows and 21 columns, with no missing values. Each row is an **order line**: there are 5,111 distinct orders, and Row ID is unique. Orders run from 3 January 2023 to 30 December 2026 with every month present. 1,093 lines are dated after the analysis date of 15 October 2026; they are flagged, not removed.
- **US rows are internally consistent.** They pass every completeness, consistency and value-range check: one date, customer and location per order; IDs that match their patterns; and a single price per product.
- **Canadian rows are excluded** (200 rows, 102 orders, 1.26% of sales). 74 of them (37%) carry another product's price, against 0 US lines. 14 lines in 6 orders copy US orders line for line. 29 of 63 Canadian order numbers reappear in another year, against 0 for the US. Two orders change Customer ID from line to line, so Harry Olson has 5 IDs. Left in, they would raise Fasteners sales by 182%, mostly through one $5,199.96 line of Staples, and make Chairs rather than Phones the top sub-category.
- **Fixes:**
  - 438 US ZIP codes had lost their leading zero (e.g. Linden, NJ 7036 → 07036) in Connecticut, Maine, Massachusetts, New Hampshire, New Jersey and Rhode Island.
  - 3 Customer IDs (43 lines) were upper-cased.
  - 7 product names had repeated spaces collapsed.
  - 2 exact repeat lines were removed (Row IDs 392 and 1700).
- **Known but kept:**
  - 32 Product IDs cover more than one product name, so product analysis uses ID + name.
  - 296 orders (800 lines) are in the Returns sheet and are flagged with `is_returned`.
- **Result:** the cleaned file has 10,192 rows × 28 columns. The **analysis set** has 9,993 US lines, 5,009 orders and 793 customers, of which 1,069 lines are dated after 15 October 2026. Every raw line is accounted for, and the totals reconcile exactly.